# Aula 5 – MLP em Keras: Experimentos, LOSS, Baseline e Overfitting

## Objetivos de aprendizado

- Revisar a MLP construída na aula passada (Aula 4).
- Entender, na prática, o que é **LOSS** (função de perda) e por que ela é importante.
- Entender o conceito de **modelo baseline**.
- Observar como **neurônios, camadas e epochs** afetam treino e validação.
- Identificar sinais de **underfitting** e **overfitting** nos gráficos.

## Contexto

Você continua como analista de dados da loja online. Seu modelo de MLP prevê se um cliente vai **comprar** (1) ou **não comprar** (0).

Hoje, seu chefe quer que você ajuste melhor esse modelo, mas **sem decorar os dados de treino** e mantendo o treinamento em um tempo razoável.

---

## O que é um **baseline**?

Antes de começar a “tunar” (ajustar) o modelo, precisamos de um ponto de referência.

- **Baseline** é um modelo inicial simples, que serve como **referência mínima**.
- A ideia é: *“Qual o desempenho que eu preciso superar para dizer que melhorei alguma coisa?”*

Exemplos de baseline:
- Em classificação: sempre prever a classe mais frequente.
- Em redes neurais: uma MLP bem simples com poucos neurônios e poucas camadas.

Na aula de hoje, vamos definir uma MLP simples como nosso **modelo baseline** e, a partir dele, fazer experimentos para ver o que melhora ou piora.

## 1. Setup inicial

Nesta aula vamos:
- Gerar um dataset sintético simples de clientes (para manter tudo no mesmo notebook).
- Separar em treino / validação / teste.
- Padronizar as features.

> Se na Aula 4 você usou um dataset real (por exemplo, `clientes.csv`), pode adaptar esta parte trocando o `df` gerado aqui pelo `df` da aula anterior.

In [ ]:
# Imports básicos
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from tensorflow import keras
from tensorflow.keras import layers

sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (8, 4)

### 1.1. Gerando um dataset sintético de clientes

Vamos criar um conjunto de dados simples com algumas features:
- `tempo_site` (minutos no site)
- `paginas_vistas`
- `valor_medio_carrinho`
- `renda_mensal` (em milhares de reais)

E uma variável alvo binária `comprou` (0/1).

A regra será **aproximadamente lógica**, mas com ruído (como dados reais), de forma que:
- quanto mais tempo no site,
- mais páginas vistas,
- maior valor médio do carrinho,
- e maior renda,

maior a probabilidade de **compra**.

In [ ]:
np.random.seed(42)

n_samples = 2000

tempo_site = np.random.exponential(scale=5, size=n_samples) + 1   # em minutos
paginas_vistas = np.random.poisson(lam=5, size=n_samples) + 1
valor_medio_carrinho = np.random.gamma(shape=2., scale=50., size=n_samples)
renda_mensal = np.random.normal(loc=5, scale=2, size=n_samples)  # em milhares

# Regra aproximada para compra (probabilidade)
logit = (
    0.3 * tempo_site +
    0.4 * paginas_vistas +
    0.02 * valor_medio_carrinho +
    0.1 * renda_mensal -
    6.0
)
prob_compra = 1 / (1 + np.exp(-logit))

# Gera 0/1 a partir da probabilidade
comprou = (np.random.rand(n_samples) < prob_compra).astype(int)

df = pd.DataFrame({
    'tempo_site': tempo_site,
    'paginas_vistas': paginas_vistas,
    'valor_medio_carrinho': valor_medio_carrinho,
    'renda_mensal': renda_mensal,
    'comprou': comprou
})

df.head()

In [ ]:
# Distribuicao da Classe
df['comprou'].value_counts(normalize=True)

In [ ]:
# Gráfico de Contagem
sns.countplot(x='comprou', data=df)
plt.title('Distribuição da variável alvo (comprou)')
plt.show()


### 1.2. Separando treino / validação / teste e padronizando

- Treino: 60%
- Validação: 20%
- Teste: 20%

Vamos padronizar as features (média 0, desvio 1) para ajudar a MLP a treinar melhor.

> **Importante:** o scaler é ajustado **só no treino** e depois aplicado em validação e teste, para não vazar informação.

In [ ]:
X = df.drop('comprou', axis=1).values
y = df['comprou'].values

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

X_train.shape, X_val.shape, X_test.shape

## 2. LOSS: o que é e como vamos observar

Durante o treino, vamos olhar duas coisas principais:
- **LOSS de treino**: erro médio nos dados que o modelo está usando para aprender.
- **LOSS de validação**: erro médio em dados **novos** (que o modelo não usa para atualizar os pesos).

### Intuição rápida de LOSS

- Imagine um jogo de dardos:
  - O **alvo** é o valor real (0 ou 1: não comprou / comprou).
  - O **dardo** é a previsão do modelo (uma probabilidade entre 0 e 1).
  - A **LOSS** mede o quão longe, em média, o dardo cai do centro.

> Treinar a rede = ajustar os pesos para diminuir essa distância média (a LOSS).

### Por que não usamos só accuracy para treinar?

- A **accuracy** olha apenas se a previsão final (0 ou 1) bateu com o valor real.
- A **LOSS** olha mais fundo: *o quão errado* ou *o quão certo* estamos, usando valores contínuos.
- Os algoritmos de otimização (como gradiente descendente) precisam de uma função **contínua** como a LOSS para saber em que direção ajustar os pesos.

Vamos criar uma função para plotar as curvas de LOSS e accuracy durante o treino, para visualizar tudo isso.

In [ ]:
# FUNCAO PLOT_HISTORY
def plot_history(history, title='Modelo'):
    """Plota as curvas de LOSS e Accuracy para treino e validação."""
    hist = history.history
    epochs = range(1, len(hist['loss']) + 1)

    plt.figure(figsize=(12, 4))

    # LOSS
    plt.subplot(1, 2, 1)
    plt.plot(epochs, hist['loss'], 'b-', label='Treino (loss)')
    plt.plot(epochs, hist['val_loss'], 'r--', label='Validação (loss)')
    plt.xlabel('Epoch')
    plt.ylabel('LOSS')
    plt.title(f'{title} - LOSS')
    plt.legend()

    # ACCURACY
    plt.subplot(1, 2, 2)
    plt.plot(epochs, hist['accuracy'], 'b-', label='Treino (acc)')
    plt.plot(epochs, hist['val_accuracy'], 'r--', label='Validação (acc)')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title(f'{title} - Accuracy')
    plt.legend()

    plt.show()

## 3. Nosso modelo **baseline** (MLP simples)

Vamos definir uma MLP bem simples como **baseline**:
- 1 camada oculta com 16 neurônios e ativação **ReLU**.
- 1 neurônio de saída com ativação **sigmóide** (para previsão de probabilidade entre 0 e 1).
- Otimizador: **Adam**.
- Função de LOSS: **`binary_crossentropy`** (clássica para classificação binária).

> Lembre: **baseline** é o nosso modelo de referência. Tudo o que fizermos depois deve ser comparado com ele.

In [ ]:
def build_model_baseline(input_dim):
    """Cria o modelo baseline: MLP simples com 1 camada oculta."""
    model = keras.Sequential([
        layers.Dense(16, activation='relu', input_shape=(input_dim,)),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',  # <<< FUNÇÃO DE LOSS
        metrics=['accuracy']
    )
    return model

baseline_model = build_model_baseline(X_train.shape[1])

history_baseline = baseline_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)

plot_history(history_baseline, title='Baseline (16 neurônios, 1 camada)')

### Interpretação: LOSS de treino x LOSS de validação

Observe o gráfico de **LOSS**:
- A curva **azul** é o erro médio nos **dados de treino**.
- A curva **vermelha** é o erro médio nos **dados de validação**.

**Responda (mentalmente ou escrevendo abaixo):**
1. As duas curvas de LOSS estão descendo? Elas estão relativamente próximas?
2. Você vê algum ponto onde a LOSS de validação começa a subir enquanto a de treino continua caindo?
3. A partir desse gráfico, você diria que o modelo está **ok**, com **underfitting** ou com **overfitting**?

> Dica: **overfitting** aparece quando a perda de treino continua caindo, mas a de validação começa a subir (modelo decorando o treino).

Você pode escrever aqui sua interpretação em forma de comentário:

In [ ]:
# Espaço para você escrever (opcional)
comentario_baseline = """Escreva aqui, em 1 ou 2 frases, se você acha que o baseline
está mais para underfitting, overfitting ou um meio-termo aceitável, com base nas curvas de LOSS.
"""
print(comentario_baseline)

## 4. Experimento 1 – Aumentando o número de neurônios (Largura)

Agora vamos **dobrar** o número de neurônios da camada oculta:
- De 16 → 64 neurônios.

O que você espera?
- Mais neurônios → mais capacidade de aprender padrões complexos.
- Mas também → maior risco de **overfitting**.

Vamos treinar e comparar com o baseline.

In [ ]:
# MODELO WIDE
def build_model_wide(input_dim):
    """Modelo com camada oculta mais larga (mais neurônios)."""
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

wide_model = build_model_wide(X_train.shape[1])

history_wide = wide_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)

plot_history(history_wide, title='Modelo Largo (64 neurônios, 1 camada)')

### Desafio 1 – Compare com o baseline

Compare o **baseline** (16 neurônios) com o modelo **largo** (64 neurônios):

- A **LOSS de treino** (azul) ficou menor com 64 neurônios?
- E a **LOSS de validação** (vermelha)? Melhorou ou piorou?
- O **gap** entre treino e validação aumentou ou diminuiu?

Se o gap aumentou (treino bem melhor que validação), é um sinal de que a rede está com mais tendência a **overfitting**.

Escreva em 1 frase o que você observou (opcional):

In [ ]:
comentario_largo = """Escreva aqui sua conclusão sobre o modelo largo vs baseline.
"""
print(comentario_largo)

## 5. Experimento 2 – Mais uma camada oculta (Profundidade)

Agora vamos testar uma MLP com **duas camadas ocultas**:
- 1ª camada: 32 neurônios (ReLU)
- 2ª camada: 16 neurônios (ReLU)
- Saída: 1 neurônio (sigmóide)

Isso aumenta ainda mais a capacidade da rede – e também o risco de **overfitting**.

Vamos treinar e observar as curvas de LOSS e accuracy.

In [ ]:
#MODELO DEEP
def build_model_deeper(input_dim):
    """Modelo mais profundo (duas camadas ocultas)."""
    model = keras.Sequential([
        layers.Dense(32, activation='relu', input_shape=(input_dim,)),
        layers.Dense(16, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

deep_model = build_model_deeper(X_train.shape[1])

history_deep = deep_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)

plot_history(history_deep, title='Modelo Profundo (2 camadas ocultas)')

### Desafio 2 – Identificando overfitting

Compare com o **baseline**:

- As curvas de **LOSS** (treino/validação) ficaram melhores ou piores?
- Você vê algum ponto (epoch) em que a **val_loss** começa a subir enquanto a loss de treino continua caindo?

Se isso acontecer, é o **sinal clássico de overfitting**:
> O modelo está aprendendo demais os detalhes do treino e piorando em dados novos.

Anote sua observação (opcional):

In [ ]:
comentario_profundo = """Escreva aqui sua conclusão sobre o modelo profundo vs baseline.
"""
print(comentario_profundo)

## 6. Experimento 3 – Quantas epochs treinar?

Vamos usar o **mesmo modelo baseline**, mas variar o número de epochs:
- 10 epochs
- 30 epochs
- 100 epochs

Ideia: ver a partir de qual ponto treinar mais **só piora a validação** (overfitting).

> Repare principalmente na **val_loss**: se ela começa a subir, pode ser melhor parar o treino antes (técnica chamada *early stopping*).

In [ ]:
def train_with_epochs(epochs, title_prefix='Baseline'):
    model = build_model_baseline(X_train.shape[1])
    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=32,
        validation_data=(X_val, y_val),
        verbose=0
    )
    plot_history(history, title=f'{title_prefix} ({epochs} epochs)')
    return history

history_10  = train_with_epochs(10,  'Baseline')
history_30  = train_with_epochs(30,  'Baseline')
history_100 = train_with_epochs(100, 'Baseline')

### Desafio 3 – Encontrando o ponto “bom o suficiente”

Observe principalmente o caso com **100 epochs**:

- A **LOSS de treino** continua caindo quase sempre.
- A **LOSS de validação** em algum momento começa a **subir**.

Quando isso acontece?
- Esse é o ponto em que o modelo começa a **decorar** o treino.
- Na prática, poderíamos **parar o treino antes** (técnica de *early stopping*).

Pergunta para refletir:
- Se você tivesse limite de tempo de treino (por exemplo, só pode treinar por 20 epochs), qual configuração (número de neurônios, camadas etc.) você escolheria para ter um modelo **bom o suficiente**, comparando com o baseline?

## 7. Experimento 4 – Dropout para reduzir overfitting (opcional)

Agora vamos adicionar **Dropout**, uma técnica simples de regularização.

Ideia do Dropout:
- Durante o treino, desligar aleatoriamente uma porcentagem de neurônios.
- Isso força a rede a não depender demais de poucos neurônios específicos.
- Na média, ajuda a reduzir **overfitting**.

Vamos usar:
- Duas camadas densas (64 e 32 neurônios).
- Dropout de 50% entre elas.

> Se estiver sem tempo em aula, esta parte pode virar um **desafio extra** ou atividade para casa.

In [ ]:
def build_model_dropout(input_dim):
    """Modelo com Dropout para reduzir overfitting."""
    model = keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.5),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

drop_model = build_model_dropout(X_train.shape[1])

history_drop = drop_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    verbose=0
)

plot_history(history_drop, title='Modelo com Dropout (64-32 neurônios)')

### Desafio 4 – Dropout funcionou?

Compare o modelo **com Dropout** com o modelo **largo sem Dropout** (64 neurônios):

- A **LOSS de treino** com Dropout pode ficar um pouco pior (mais alta).
- Mas a **LOSS de validação** melhorou? Ficou mais estável? O gap entre treino e validação diminuiu?

Se sim, é sinal de que o Dropout ajudou a combater o **overfitting**.

> Moral da história: às vezes, aceitar um treino um pouco pior (LOSS maior no treino) pode trazer um desempenho **melhor em dados novos**.

Anote sua observação (opcional):

In [ ]:
comentario_dropout = """Escreva aqui sua conclusão sobre o efeito do Dropout.
"""
print(comentario_dropout)

## 8. Avaliando no conjunto de teste

Vamos escolher **um modelo** (por exemplo, o com Dropout) e avaliar no conjunto de teste, que o modelo nunca viu.

Aqui vou usar o **modelo com Dropout**, mas você pode repetir com qualquer um dos outros modelos e comparar.

> Lembre: o desempenho no **teste** é o que mais se aproxima do que aconteceria em produção (dados realmente novos).

In [ ]:
test_loss, test_acc = drop_model.evaluate(X_test, y_test, verbose=0)
print(f'LOSS no teste: {test_loss:.4f}')
print(f'Accuracy no teste: {test_acc:.4f}')

## 9. Resumo da Aula 5

Hoje você:

- Viu que **LOSS** é a medida principal de erro que o modelo tenta **minimizar** durante o treinamento.
- Aprendeu a olhar **LOSS de treino x LOSS de validação** para identificar:
  - **Underfitting**: LOSS alta em tudo, modelo não aprende o padrão.
  - **Overfitting**: treino melhora (LOSS caindo), mas validação piora (LOSS subindo).
- Definiu e usou um **modelo baseline** (MLP simples) como referência.
- Brincou com a arquitetura da MLP:
  - Número de neurônios (largura).
  - Número de camadas (profundidade).
  - Número de epochs (tempo de treino).
  - Dropout (regularização para reduzir overfitting).

### Perguntas para reflexão

1. Como você explicaria **LOSS** em **uma frase** para alguém da área de negócios?
2. Se o seu modelo tem **val_loss** muito maior que **loss**, o que você tentaria fazer para melhorar?
3. O modelo com mais neurônios ou mais camadas foi realmente **melhor** que o baseline? Em que aspecto?

### Próxima aula

Na próxima aula, vamos abrir a caixa-preta:
- Como o modelo realmente **ajusta os pesos** para reduzir a LOSS?
- Intuição de **Gradiente Descendente** e **Backpropagation**, conectando com os slides de conceitos fundamentais vistos no início da disciplina.